# Proyecto Big Data – Consultas Q1–Q10 con Apache Spark (Dataproc)
**UTEC · Big Data (DS4341)** · Clúster `cluster-ef56` · Kernel: **PySpark**

Dataset: GeoNames, leído desde `gs://bucket_test_data_processing/allCountries_headers.csv`
(10,330,862 registros; extracto que no incluye los países con código ISO posterior a `TH`, p. ej. US, UY, VE).

Cada celda ejecuta una consulta y mide su tiempo con `time.time()`. En Q1–Q4 se usa `cache()` + `count()`
para que Spark ejecute la consulta en su propia celda (evaluación perezosa). La última celda resume todos los tiempos.

In [ ]:
import time
import pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.sparkContext.setLogLevel("ERROR")   # oculta los WARN
tiempos = {}   # aquí se guarda cuánto tarda cada consulta

esquema = """geonameid LONG, name STRING, asciiname STRING, alternatenames STRING,
             latitude DOUBLE, longitude DOUBLE, fclass STRING, fcode STRING, country STRING,
             cc2 STRING, admin1 STRING, admin2 STRING, admin3 STRING, admin4 STRING,
             population LONG, elevation DOUBLE, dem LONG, timezone STRING, moddate STRING"""
RUTA = "gs://bucket_test_data_processing/allCountries_headers.csv"

# ---------------------------------------------------------------- carga (cronometrada)
spark.catalog.clearCache()   # borra datos cacheados de corridas anteriores -> carga real desde el bucket
t0 = time.time()
sdf = (spark.read.option("header", True).option("quote", '"').option("escape", '"')
            .schema(esquema).csv(RUTA)
            .select("geonameid", "name", "latitude", "longitude", "fclass", "fcode", "country",
                    "admin1", "population", "elevation", "dem", "timezone", "moddate")
            .cache())
n_registros = sdf.count()
tiempos["Carga"] = time.time() - t0

# ---------------------------------------------------------------- resumen
print("=" * 60)
print("  CARGA DEL DATASET EN SPARK")
print("=" * 60)
print(f"  Origen          : {RUTA}")
print(f"  Registros       : {n_registros:>12,}")
print(f"  Columnas        : {len(sdf.columns):>12}")
print(f"  Particiones     : {sdf.rdd.getNumPartitions():>12}")
print(f"  Tiempo de carga : {tiempos['Carga']:>10.2f} s")
print("=" * 60)

# ---------------------------------------------------------------- equivalente a df.info()
no_nulos = sdf.select([F.count(c).alias(c) for c in sdf.columns]).first().asDict()
info = pd.DataFrame({
    "columna": sdf.columns,
    "tipo": [t for _, t in sdf.dtypes],
    "no_nulos": [no_nulos[c] for c in sdf.columns],
})
info["nulos"] = n_registros - info["no_nulos"]
info["% nulos"] = (info["nulos"] / n_registros * 100).round(2)
info.index = range(1, len(info) + 1)
display(info.style.format({"no_nulos": "{:,}", "nulos": "{:,}", "% nulos": "{:.2f}%"}))

## Q1 – Limpieza y tipado

In [ ]:
t0 = time.time()

# Cuántos registros afecta cada regla de limpieza (antes de aplicarla)
cambios = sdf.select(
    F.sum((F.col("name") != F.trim("name")).cast("int")).alias("nombres_con_espacios"),
    F.sum(F.col("population").isNull().cast("int")).alias("poblacion_nula"),
    F.sum((F.col("dem") == -9999).cast("int")).alias("dem_sin_dato"),
).first().asDict()

# Limpieza: quitar espacios del nombre, población vacía -> 0, DEM -9999 -> nulo, texto -> fecha
nuevo = (sdf.withColumn("name", F.trim("name"))
            .withColumn("population", F.coalesce("population", F.lit(0)))
            .withColumn("dem", F.when(F.col("dem") == -9999, None).otherwise(F.col("dem")))
            .withColumn("moddate", F.to_date("moddate", "yyyy-MM-dd"))
            .cache())
n = nuevo.count()   # obliga a Spark a ejecutar la limpieza ahora
sdf.unpersist(); sdf = nuevo
tiempos["Q1"] = time.time() - t0

print("=" * 60)
print("  Q1 - LIMPIEZA Y TIPADO")
print("=" * 60)
print(f"  Nombres con espacios (trim)   : {cambios['nombres_con_espacios']:>12,}")
print(f"  Población nula -> 0           : {cambios['poblacion_nula']:>12,}")
print(f"  DEM -9999 -> nulo             : {cambios['dem_sin_dato']:>12,}")
print(f"  moddate: texto -> fecha       : {'OK':>12}")
print(f"  Registros después             : {n:>12,}")
print(f"  Tiempo                        : {tiempos['Q1']:>10.2f} s")
print("=" * 60)

## Q2 – Tratamiento de duplicados

In [ ]:
t0 = time.time()

n0 = sdf.count()
# 1) Duplicados exactos por identificador
sin_dup_id = sdf.dropDuplicates(["geonameid"])
n1 = sin_dup_id.count()
# 2) Duplicados lógicos: mismo nombre + país + coordenadas -> se conserva el menor geonameid
conservar = (sin_dup_id.groupBy("name", "country", "latitude", "longitude")
                       .agg(F.min("geonameid").alias("geonameid")))
nuevo = sin_dup_id.join(conservar.select("geonameid"), on="geonameid", how="left_semi").cache()
n2 = nuevo.count()
sdf.unpersist(); sdf = nuevo
tiempos["Q2"] = time.time() - t0

print("=" * 60)
print("  Q2 - TRATAMIENTO DE DUPLICADOS")
print("=" * 60)
print(f"  Registros antes                     : {n0:>12,}")
print(f"  Duplicados por geonameid            : {n0 - n1:>12,}")
print(f"  Duplicados nombre+país+coordenadas  : {n1 - n2:>12,}")
print(f"  Registros después                   : {n2:>12,}")
print(f"  Tiempo                              : {tiempos['Q2']:>10.2f} s")
print("=" * 60)

## Q3 – Validación y eliminación (DELETE) de registros inválidos

In [ ]:
t0 = time.time()

reglas = {
    "Latitud fuera de [-90, 90]"    : ~F.col("latitude").between(-90, 90),
    "Longitud fuera de [-180, 180]" : ~F.col("longitude").between(-180, 180),
    "Población negativa"            : F.col("population") < 0,
    "País nulo"                     : F.col("country").isNull(),
    "Nombre nulo"                   : F.col("name").isNull(),
}
# Cuántos registros incumple cada regla
conteos = sdf.select([F.sum(F.coalesce(e, F.lit(False)).cast("int")).alias(f"r{i}")
                      for i, e in enumerate(reglas.values())]).first()

# Eliminar los que incumplen al menos una regla
invalido = None
for e in reglas.values():
    e = F.coalesce(e, F.lit(False))
    invalido = e if invalido is None else (invalido | e)
antes = sdf.count()
nuevo = sdf.filter(~invalido).cache()
despues = nuevo.count()
sdf.unpersist(); sdf = nuevo
tiempos["Q3"] = time.time() - t0

print("=" * 60)
print("  Q3 - VALIDACIÓN Y ELIMINACIÓN (DELETE)")
print("=" * 60)
for (nombre, _), c in zip(reglas.items(), conteos):
    print(f"  {nombre:<34}: {c:>12,}")
print("-" * 60)
print(f"  Registros antes                   : {antes:>12,}")
print(f"  Registros eliminados              : {antes - despues:>12,}")
print(f"  Registros después                 : {despues:>12,}")
print(f"  Tiempo                            : {tiempos['Q3']:>10.2f} s")
print("=" * 60)

## Q4 – Transformación de variables (CREATE)

In [ ]:
t0 = time.time()

clases = {"A": "Division administrativa", "H": "Hidrografia (rios, lagos)", "L": "Areas y parques",
          "P": "Ciudades y poblados", "R": "Carreteras y vias ferreas", "S": "Edificios y puntos de interes",
          "T": "Relieve (montanas, cerros)", "U": "Submarino", "V": "Vegetacion y bosques"}
mapa = F.create_map([F.lit(x) for kv in clases.items() for x in kv])
pop = F.col("population")

nuevo = (sdf.withColumn("anio_mod", F.year("moddate"))
            .withColumn("hemisferio", F.when(F.col("latitude") >= 0, "Norte").otherwise("Sur"))
            .withColumn("elevacion_final", F.coalesce(F.col("elevation"), F.col("dem").cast("double")))
            .withColumn("poblacion_asentamiento", F.when(F.col("fclass") == "P", pop).otherwise(F.lit(0)))
            .withColumn("fclass_desc", F.coalesce(mapa[F.col("fclass")], F.lit("Desconocido")))
            .withColumn("categoria_poblacion",
                        F.when(pop <= 0, "Sin poblacion").when(pop < 1_000, "< 1K")
                         .when(pop < 100_000, "1K - 100K").when(pop < 1_000_000, "100K - 1M")
                         .otherwise("> 1M"))
            .cache())
n = nuevo.count()
sdf.unpersist(); sdf = nuevo
categorias = (sdf.groupBy("categoria_poblacion").count()
                 .orderBy(F.desc("count")).toPandas())
tiempos["Q4"] = time.time() - t0

print("=" * 60)
print("  Q4 - TRANSFORMACIÓN DE VARIABLES (CREATE)")
print("=" * 60)
print("  Columnas nuevas:")
print("    anio_mod               <- año de moddate")
print("    hemisferio             <- Norte / Sur según latitud")
print("    elevacion_final        <- elevation o, si falta, dem")
print("    poblacion_asentamiento <- población solo si es ciudad (P)")
print("    fclass_desc            <- descripción de la clase")
print("    categoria_poblacion    <- rango de población")
print("-" * 60)
print("  Registros por categoría de población:")
for _, fila in categorias.iterrows():
    print(f"    {fila['categoria_poblacion']:<16}: {fila['count']:>12,}  ({fila['count'] / n * 100:5.2f}%)")
print("-" * 60)
print(f"  Registros / columnas        : {n:,} / {len(sdf.columns)}")
print(f"  Tiempo                      : {tiempos['Q4']:>10.2f} s")
print("=" * 60)

## Q5 – Filtrado + ordenamiento: centros poblados del Perú

In [ ]:
t0 = time.time()

peru = sdf.filter((F.col("country") == "PE") & (F.col("fclass") == "P") & (F.col("population") > 0))
n_peru = peru.count()
top20 = (peru.orderBy(F.desc("population"))
             .select("name", "admin1", "population", "elevacion_final", "latitude", "longitude")
             .limit(20).toPandas())
tiempos["Q5"] = time.time() - t0

print("=" * 60)
print("  Q5 - FILTRADO + ORDENAMIENTO: CIUDADES DEL PERÚ")
print("=" * 60)
print(f"  Filtro      : country = PE, fclass = P, population > 0")
print(f"  Resultados  : {n_peru:,} centros poblados con población")
print(f"  Tiempo      : {tiempos['Q5']:.2f} s")
print("=" * 60)
top20.index = range(1, len(top20) + 1)
display(top20.style.format({"population": "{:,}", "elevacion_final": "{:,.0f}",
                            "latitude": "{:.4f}", "longitude": "{:.4f}"}))

## Q6 – Agregación por país

In [ ]:
t0 = time.time()

por_pais = (sdf.groupBy("country")
               .agg(F.count("*").alias("registros"),
                    F.sum("poblacion_asentamiento").alias("poblacion_total"),
                    F.round(F.avg("elevacion_final"), 2).alias("elevacion_promedio"))
               .orderBy(F.desc("poblacion_total"))
               .toPandas())
tiempos["Q6"] = time.time() - t0

print("=" * 60)
print("  Q6 - AGREGACIÓN POR PAÍS")
print("=" * 60)
print(f"  Agrupado por : country")
print(f"  Métricas     : registros, población en asentamientos, elevación promedio")
print(f"  Países       : {len(por_pais):,}")
print(f"  Tiempo       : {tiempos['Q6']:.2f} s")
print("=" * 60)
print("  Top 15 países por población en asentamientos (fclass = P):")
top15 = por_pais.head(15).copy()
top15.index = range(1, len(top15) + 1)
display(top15.style.format({"registros": "{:,}", "poblacion_total": "{:,}", "elevacion_promedio": "{:,.2f}"}))

## Q7 – Agrupación por clase de entidad

In [ ]:
t0 = time.time()

total = sdf.count()
por_clase = (sdf.groupBy("fclass", "fclass_desc").count()
                .withColumnRenamed("count", "registros")
                .withColumn("porcentaje", F.round(F.col("registros") / total * 100, 2))
                .orderBy(F.desc("registros"))
                .toPandas())
tiempos["Q7"] = time.time() - t0

print("=" * 60)
print("  Q7 - AGRUPACIÓN POR CLASE DE ENTIDAD")
print("=" * 60)
for _, f in por_clase.iterrows():
    clase = f["fclass"] if pd.notna(f["fclass"]) else "-"
    print(f"  {clase:<2} {f['fclass_desc']:<32}: {f['registros']:>10,}  ({f['porcentaje']:5.2f}%)")
print("-" * 60)
print(f"  Total       : {total:,}")
print(f"  Tiempo      : {tiempos['Q7']:.2f} s")
print("=" * 60)

## Q8 – Ciudades de más de 1 millón de habitantes por país

In [ ]:
t0 = time.time()

grandes = (sdf.filter((F.col("fclass") == "P") & (F.col("population") > 1_000_000))
              .groupBy("country")
              .agg(F.count("*").alias("ciudades_mas_1M"),
                   F.sum("population").alias("poblacion_en_esas_ciudades"))
              .orderBy(F.desc("ciudades_mas_1M"), F.desc("poblacion_en_esas_ciudades"))
              .toPandas())
tiempos["Q8"] = time.time() - t0

print("=" * 60)
print("  Q8 - CIUDADES DE MÁS DE 1 MILLÓN POR PAÍS")
print("=" * 60)
print(f"  Filtro      : fclass = P, population > 1,000,000")
print(f"  Países      : {len(grandes):,}")
print(f"  Ciudades    : {grandes['ciudades_mas_1M'].sum():,}")
print(f"  Tiempo      : {tiempos['Q8']:.2f} s")
print("=" * 60)
top15 = grandes.head(15).copy()
top15.index = range(1, len(top15) + 1)
display(top15.style.format({"ciudades_mas_1M": "{:,}", "poblacion_en_esas_ciudades": "{:,}"}))

## Q9 – Top 3 ciudades por país de Sudamérica (ranking con función de ventana)

In [ ]:
t0 = time.time()

sudamerica = ["AR", "BO", "BR", "CL", "CO", "EC", "GY", "PE", "PY", "SR", "UY", "VE"]
w = Window.partitionBy("country").orderBy(F.desc("population"))
ranking = (sdf.filter(F.col("country").isin(sudamerica) & (F.col("fclass") == "P"))
              .withColumn("ranking", F.row_number().over(w))
              .filter(F.col("ranking") <= 3)
              .select("country", "ranking", "name", "population")
              .orderBy("country", "ranking")
              .toPandas())
tiempos["Q9"] = time.time() - t0

presentes = sorted(ranking["country"].unique())
faltantes = [p for p in sudamerica if p not in presentes]
print("=" * 60)
print("  Q9 - TOP 3 CIUDADES POR PAÍS DE SUDAMÉRICA (RANKING)")
print("=" * 60)
print(f"  Técnica     : función de ventana row_number() por país")
print(f"  Países      : {len(presentes)} de {len(sudamerica)}")
if faltantes:
    print(f"  Sin datos   : {', '.join(faltantes)} (no están en el CSV truncado)")
print(f"  Tiempo      : {tiempos['Q9']:.2f} s")
print("=" * 60)
ranking.index = range(1, len(ranking) + 1)
display(ranking.style.format({"population": "{:,}"}))

## Q10 – Registros modificados por año y % acumulado

In [ ]:
t0 = time.time()

w_acum = Window.orderBy("anio_mod").rowsBetween(Window.unboundedPreceding, 0)
por_anio = (sdf.filter(F.col("anio_mod").isNotNull())
               .groupBy("anio_mod").count()
               .withColumnRenamed("count", "registros")
               .withColumn("pct_acumulado",
                           F.round(F.sum("registros").over(w_acum)
                                   / F.sum("registros").over(Window.partitionBy()) * 100, 2))
               .orderBy("anio_mod")
               .toPandas())
tiempos["Q10"] = time.time() - t0

print("=" * 60)
print("  Q10 - REGISTROS MODIFICADOS POR AÑO Y % ACUMULADO")
print("=" * 60)
for _, f in por_anio.iterrows():
    print(f"  {int(f['anio_mod'])} : {f['registros']:>10,}   acumulado {f['pct_acumulado']:6.2f}%")
print("-" * 60)
print(f"  Años        : {len(por_anio)} ({int(por_anio['anio_mod'].min())}–{int(por_anio['anio_mod'].max())})")
print(f"  Tiempo      : {tiempos['Q10']:.2f} s")
print("=" * 60)

## Resumen de tiempos (benchmark de Spark)

In [ ]:
import matplotlib.pyplot as plt

descripcion = {"Carga": "Lectura del CSV desde GCS", "Q1": "Limpieza y tipado", "Q2": "Duplicados",
               "Q3": "Validación (DELETE)", "Q4": "Transformación de variables", "Q5": "Filtrado Perú",
               "Q6": "Agregación por país", "Q7": "Agrupación por clase", "Q8": "Ciudades > 1M",
               "Q9": "Top 3 Sudamérica", "Q10": "Registros por año"}
resumen = pd.DataFrame([(k, descripcion.get(k, ""), v) for k, v in tiempos.items()],
                       columns=["paso", "descripcion", "segundos"])
total_q = resumen.loc[resumen["paso"].str.startswith("Q"), "segundos"].sum()
resumen["% del total"] = resumen["segundos"] / resumen["segundos"].sum() * 100

print("=" * 60)
print("  RESUMEN DE TIEMPOS - APACHE SPARK")
print("=" * 60)
for _, f in resumen.iterrows():
    print(f"  {f['paso']:<6} {f['descripcion']:<28}: {f['segundos']:>8.2f} s  ({f['% del total']:5.1f}%)")
print("-" * 60)
print(f"  Total 10 consultas                  : {total_q:>8.2f} s")
print(f"  Total con carga                     : {resumen['segundos'].sum():>8.2f} s")
print(f"  Consulta más lenta                  : {resumen.loc[resumen['paso'].str.startswith('Q')].sort_values('segundos').iloc[-1]['paso']}")
print(f"  Throughput (registros/s, total)     : {n_registros / resumen['segundos'].sum():>12,.0f}")
print("=" * 60)

resumen.round(2).to_csv("tiempos_spark.csv", index=False)
print("Guardado: tiempos_spark.csv")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(resumen["paso"], resumen["segundos"], color="#2a78d6")
for i, v in enumerate(resumen["segundos"]):
    ax.text(v, i, f" {v:.1f} s", va="center", fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("segundos")
ax.set_title("Apache Spark en Dataproc – tiempo por paso")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig("tiempos_spark.png", dpi=150); plt.show()